# Simulation

# Coins simulation
## ** Using guessed coefficients

# Linear regression simulation
## ** Using guessed coefficients
## ** Using bootstrap

# Coins simulation

In [1]:
from IPython.display import display, HTML

display(HTML("""
<style>
.jp-OutputArea-output {
    max-height: none !important;
}

.jp-OutputArea-child {
    max-height: none !important;
}

.output_scroll {
    height: auto !important;
    max-height: none !important;
    overflow-y: visible !important;
}

.output_wrapper, .output {
    height: auto !important;
    max-height: none !important;
}
</style>
"""))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from ipywidgets import interact, IntSlider, FloatSlider
from math import comb


def coin_flip_simulator(
    num_coins=10,
    num_times=20,
    prob_heads=0.5,
    threshold_c=2.0,
    random_seed=42
):
    rng = np.random.default_rng(random_seed)

    flips = rng.binomial(
        n=1,
        p=prob_heads,
        size=(num_times, num_coins)
    )

    heads_per_trial = flips.sum(axis=1)
    estimated_probabilities = heads_per_trial / num_coins

    mean_estimated_probability = estimated_probabilities.mean()

    sample_deviation = estimated_probabilities.std(ddof=1)

    sample_deviation_divided_by_sqrt_num_trials = (
        sample_deviation / np.sqrt(num_times)
    )

    estimated_standard_errors_per_trial = np.sqrt(
        estimated_probabilities * (1 - estimated_probabilities) / num_coins
    )

    average_estimated_standard_error_per_trial = (
        estimated_standard_errors_per_trial.mean()
    )

    estimated_overall_standard_error = np.sqrt(
        mean_estimated_probability
        * (1 - mean_estimated_probability)
        / (num_coins * num_times)
    )

    null_probability = 0.5

    null_trial_standard_deviation = np.sqrt(
        null_probability
        * (1 - null_probability)
        / num_coins
    )

    upper_threshold = (
        null_probability + threshold_c * null_trial_standard_deviation
    )

    lower_threshold = (
        null_probability - threshold_c * null_trial_standard_deviation
    )

    num_above_upper_threshold = np.sum(estimated_probabilities > upper_threshold)
    num_below_lower_threshold = np.sum(estimated_probabilities < lower_threshold)

    num_outside_thresholds = np.sum(
        (estimated_probabilities > upper_threshold)
        | (estimated_probabilities < lower_threshold)
    )

    num_inside_thresholds = np.sum(
        (estimated_probabilities <= upper_threshold)
        & (estimated_probabilities >= lower_threshold)
    )

    percent_above_upper_threshold = 100 * num_above_upper_threshold / num_times
    percent_below_lower_threshold = 100 * num_below_lower_threshold / num_times
    percent_outside_thresholds = 100 * num_outside_thresholds / num_times
    percent_inside_thresholds = 100 * num_inside_thresholds / num_times

    if num_coins <= 10 and num_times <= 20:
        fig, ax = plt.subplots(
            figsize=(max(7, num_coins * 0.8), max(4, num_times * 0.45))
        )

        for trial_idx in range(num_times):
            for coin_idx in range(num_coins):
                is_heads = flips[trial_idx, coin_idx] == 1

                circle = Circle(
                    (coin_idx, num_times - 1 - trial_idx),
                    radius=0.35,
                    facecolor="#ffd966" if is_heads else "#d9d9d9",
                    edgecolor="black",
                    linewidth=1.2
                )

                ax.add_patch(circle)

                ax.text(
                    coin_idx,
                    num_times - 1 - trial_idx,
                    "H" if is_heads else "T",
                    ha="center",
                    va="center",
                    fontsize=12,
                    fontweight="bold"
                )

        ax.set_xlim(-0.75, num_coins - 0.25)
        ax.set_ylim(-0.75, num_times - 0.25)
        ax.set_aspect("equal")
        ax.set_xticks(range(num_coins))
        ax.set_yticks(range(num_times))
        ax.set_xticklabels([f"Coin {i + 1}" for i in range(num_coins)])
        ax.set_yticklabels([f"Trial {num_times - i}" for i in range(num_times)])
        ax.set_title("Coin Flips by Trial")
        ax.grid(alpha=0.2)
        plt.xticks(rotation=45)
        plt.show()

    else:
        print(
            "Coin drawing skipped because coins are only drawn when "
            "num_coins <= 10 and num_times <= 20."
        )
        print()

    fig, ax = plt.subplots(figsize=(9, 5))

    histogram_bins = np.linspace(
        -0.5 / num_coins,
        1 + 0.5 / num_coins,
        num_coins + 2
    )

    ax.hist(
        estimated_probabilities,
        bins=histogram_bins,
        edgecolor="black",
        alpha=0.75,
        label="Simulated trial estimates"
    )

    possible_heads = np.arange(num_coins + 1)
    possible_estimated_probabilities = possible_heads / num_coins

    null_pmf = np.array([
        comb(num_coins, k)
        * (null_probability ** k)
        * ((1 - null_probability) ** (num_coins - k))
        for k in possible_heads
    ])

    null_expected_frequencies = num_times * null_pmf

    ax.bar(
        possible_estimated_probabilities,
        null_expected_frequencies,
        width=0.85 / num_coins,
        color="orange",
        edgecolor="darkorange",
        alpha=0.35,
        label="Null distribution PMF, scaled to expected frequencies"
    )

    ax.plot(
        possible_estimated_probabilities,
        null_expected_frequencies,
        color="darkorange",
        marker="o",
        linewidth=2,
        alpha=0.85,
        label="Null PMF shape"
    )

    ax.axvline(
        mean_estimated_probability,
        color="black",
        linewidth=2,
        label=f"Mean estimate = {mean_estimated_probability:.4f}"
    )

    ax.axvline(
        mean_estimated_probability - sample_deviation,
        color="red",
        linestyle="--",
        linewidth=2,
        label="Sample mean ± empirical SD of trial estimates"
    )

    ax.axvline(
        mean_estimated_probability + sample_deviation,
        color="red",
        linestyle="--",
        linewidth=2
    )

    ax.axvline(
        prob_heads,
        color="blue",
        linestyle=":",
        linewidth=2,
        label=f"True p = {prob_heads:.4f}"
    )

    ax.axvline(
        null_probability,
        color="orange",
        linestyle=":",
        linewidth=2,
        label=f"Null p = {null_probability:.4f}"
    )

    ax.axvline(
        upper_threshold,
        color="green",
        linestyle="--",
        linewidth=3,
        label=f"Null + c × theoretical null SD of p-hat = {upper_threshold:.4f}"
    )

    ax.axvline(
        lower_threshold,
        color="green",
        linestyle="--",
        linewidth=3,
        label=f"Null - c × theoretical null SD of p-hat = {lower_threshold:.4f}"
    )

    ax.set_title(
        f"Distribution of Estimated Probability of Heads\n"
        f"{num_times} trials, {num_coins} coins per trial"
    )
    ax.set_xlabel("Estimated probability of heads")
    ax.set_ylabel("Frequency")
    ax.legend()
    ax.grid(alpha=0.25)

    plt.show()

    print(f"num_coins = {num_coins}")
    print(f"num_times = {num_times}")
    print(f"true probability of heads = {prob_heads:.6f}")
    print(f"random seed = {random_seed}")
    print()
    print(f"mean estimated probability = {mean_estimated_probability:.6f}")
    print(f"sample deviation of estimated probabilities = {sample_deviation:.6f}")
    print()
    print(
        "average estimated standard error per trial "
        "sqrt(p_hat * (1 - p_hat) / num_coins) = "
        f"{average_estimated_standard_error_per_trial:.6f}"
    )
    print(
        "estimated overall standard error "
        "sqrt(p_hat * (1 - p_hat) / (num_coins * num_times)) = "
        f"{estimated_overall_standard_error:.6f}"
    )
    print()
    #print(
    #    "sample deviation of estimated probabilities divided by "
    #    "sqrt(num_times) = "
    #    f"{sample_deviation_divided_by_sqrt_num_trials:.6f}"
    #)
    #print()
    print("Frequentist-style threshold settings")
    print(f"null probability = {null_probability:.6f}")
    print(f"threshold c = {threshold_c:.6f}")
    print(
        "trial-level standard deviation under null "
        "sqrt(0.5 * 0.5 / num_coins) = "
        f"{null_trial_standard_deviation:.6f}"
    )
    print()
    print("Threshold analysis")
    print(f"lower threshold = null - cSD = {lower_threshold:.6f}")
    print(f"upper threshold = null + cSD = {upper_threshold:.6f}")
    print(f"number below lower threshold = {num_below_lower_threshold}")
    print(f"number above upper threshold = {num_above_upper_threshold}")
    print(f"number outside two-sided thresholds = {num_outside_thresholds}")
    print(f"number inside two-sided thresholds = {num_inside_thresholds}")
    print(f"percent below lower threshold = {percent_below_lower_threshold:.2f}%")
    print(f"percent above upper threshold = {percent_above_upper_threshold:.2f}%")
    print(f"percent outside two-sided thresholds = {percent_outside_thresholds:.2f}%")
    print(f"percent inside two-sided thresholds = {percent_inside_thresholds:.2f}%")
    print()
    print(
        "Interpretation: threshold_c controls the critical standard-deviation "
        "multiple around the null value p = 0.5."
    )
    print(
        "For an approximate one-sided 5% upper-tail test, use c = 1.645."
    )
    print(
        "For an approximate two-sided 5% test, use c = 1.96."
    )
    print(
        "If prob_heads is greater than 0.5, then percent above the upper "
        "threshold is an empirical one-sided power estimate."
    )
    print(
        "If prob_heads may differ in either direction from 0.5, then percent "
        "outside the lower and upper thresholds is an empirical two-sided "
        "detection rate."
    )


interact(
    coin_flip_simulator,
    num_coins=IntSlider(
        value=10,
        min=1,
        max=50,
        step=1,
        description="num_coins"
    ),
    num_times=IntSlider(
        value=20,
        min=1,
        max=100,
        step=1,
        description="num_times"
    ),
    prob_heads=FloatSlider(
        value=0.5,
        min=0.0,
        max=1.0,
        step=0.01,
        description="P(heads)"
    ),
    threshold_c=FloatSlider(
        value=2.0,
        min=0.0,
        max=5.0,
        step=0.05,
        description="c"
    ),
    random_seed=IntSlider(
        value=42,
        min=0,
        max=10000,
        step=1,
        description="seed"
    )
);

interactive(children=(IntSlider(value=10, description='num_coins', max=50, min=1), IntSlider(value=20, descrip…

In [24]:
arr = np.array([0.5, 0.6, 0.6, 0.7, 0.8])
np.mean(arr)
np.std(arr, ddof = 1)

np.float64(0.11401754250991382)

In [31]:
np.sqrt(0.64 * (1 - 0.64) / 50)

np.float64(0.06788225099390856)

In [30]:
np.mean(np.sqrt(arr * (1 - arr) / 10))

np.float64(0.14787148491472837)

In [27]:
p = 0.5
n_coins = 10
0.5 - 2 * np.sqrt(p * (1 - p) / n_coins), 0.5 + 2 * np.sqrt(p * (1 - p) / n_coins)

(np.float64(0.18377223398316206), np.float64(0.816227766016838))

In [ ]:
Variance = (value1 - mean)**2 * prob1 + (value2 - mean)**2 * prob2 + ...
(0 - P)**2 * (1 - P) +  (1 - P)**2 * P
P * (1 - P)

In [20]:
(1/2)**10 + (1/2)**10

0.001953125

In [3]:
arr = np.array([0.5] + [0.6] * 2 + [0.7] + [0.8])
print(np.mean(arr))
print(np.std(arr, ddof = 1))

0.64
0.11401754250991382


In [8]:
np.mean(np.sqrt(arr * (1 - arr) / 10))

np.float64(0.14787148491472837)

In [ ]:
Variance = (Val1 - mean)**2 * Prob1 + (Val2 - mean)**2 * Prob2 + ..
Var_Binomial = (0 - P)**2 * (1 - P) + (1 - P)**2 * P
P**2 * (1 - P) = (1 - P)**2 * P
P * (1 - P) * [P + (1 - P)]
P * (1 - P)

In [11]:
c = 2
Num_coins = 10
p_Null = 0.5
p_Null + c * np.sqrt(p_Null * (1 - p_Null) / Num_coins)
p_Null - c * np.sqrt(p_Null * (1 - p_Null) / Num_coins)

np.float64(0.18377223398316206)

In [ ]:
Var_Binomial = 0.6 * (1 - 0.6)
Var_10_Binomials_Added = 0.6 * (1 - 0.6) * 10
Var_Mean_of_10 = 0.6 * (1 - 0.6) * 10 / 100 = 0.6 * (1 - 0.6) / 10
Std_Mean_of_10 = sqrt(0.6 * (1 - 0.6) / 10)

In [9]:
arr = np.array([0.5] + [0.6] * 2 + [0.7] + [0.8])
print(np.mean(arr))
print(np.std(arr, ddof = 1))
arr

0.64
0.11401754250991382


array([0.5, 0.6, 0.6, 0.7, 0.8])

In [9]:
np.sqrt(0.64 * (1 - 0.64) / 50)

np.float64(0.06788225099390856)

In [10]:
(np.sqrt(0.5 * 0.5 / 10) + np.sqrt(0.6 * 0.4 / 10) * 2 + np.sqrt(0.7 * 0.3 / 10) + np.sqrt(0.8 * 0.2 / 10)) / 5

np.float64(0.1478714849147284)

In [ ]:
# Standard deviation of the mean of N binomials
var = (Val1 - mean)**2 * Prob1 + (Val2 - mean)**2 * Prob2
(0 - P)**2 * (1 - P) + (1 - P)**2 * P = P**2 - P**3 + P - 2 P**2 + P**3 = P - P**2 = P * (1 - P)
Var of N copies, divided by N = N * Binomial / N**2 = P * (1 - P) / N
Std of N copies divided by N = sqrt(P * (1 - P) / N)

In [95]:
arr = np.array([0.5] + [0.6] * 2 + [0.7] + [0.8])
print(np.mean(arr))
print(np.mean(arr) + np.std(arr, ddof = 1))
print(np.mean(arr) - np.std(arr, ddof = 1))

0.64
0.7540175425099138
0.5259824574900862


In [11]:
np.sqrt(0.64 * (1 - 0.64) / 50)

np.float64(0.06788225099390856)

In [98]:
# 
0.114018

np.float64(0.14787148491472837)

In [100]:
np.sqrt(0.64 * (1 - 0.64) / 50)

np.float64(0.06788225099390856)

In [101]:
#sigma_X = 0.114
#Var_X = 0.114**2
#Var_Sum_X = 0.114**2 * 5
#Var_Avg_X = 0.114**2 * 5 / 5**2 = 0.114**2 / 5
#Sigma_Avg_X = 0.114 / sqrt(5)
0.114 / np.sqrt(5)

np.float64(0.0509823498869952)

In [91]:
p = 0.5
N = 50
c = 2
sigma = np.sqrt(p * (1 - p) / N)
p + c * sigma, p - c * sigma

(np.float64(0.6414213562373094), np.float64(0.3585786437626905))

In [92]:
13/20

0.65

In [74]:
from scipy.stats import norm

# Probability of falling below +1 standard deviation
prob_below_one_std = norm.cdf(1)
print(f"CDF at +1 standard deviation: {prob_below_one_std:.4f}")

CDF at +1 standard deviation: 0.8413


In [66]:
heads = np.array([3, 3, 3, 3, 4, 1, 2, 1, 3, 2]) * 0.2
np.std(heads, ddof = 1)

np.float64(0.19436506316151006)

In [71]:
np.sqrt(0.5 * 0.5 / 50)

np.float64(0.07071067811865475)

In [73]:
p = np.std(heads, ddof = 1) / np.sqrt(10)
p
#np.sqrt(p * (1 - p) / 50)

np.float64(0.061463629715285927)

In [41]:
def factorial(n):
    if n <= 1:
        return 1
    else:
        return n * factorial(n - 1)
    
total = 0
for m in range(20, 31):
    total += int(factorial(30) / (factorial(m) * factorial(30 - m)))
total *= 0.5**30
total

0.04936857335269451

# Regression simulation

In [13]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider


def normal_pdf(x, mean, sd):
    return (
        1 / (sd * np.sqrt(2 * np.pi))
        * np.exp(-0.5 * ((x - mean) / sd) ** 2)
    )


def repeated_ols_simulator(
    num_samples=10,
    num_times=1000,
    beta_0=1.0,
    beta_1=2.0,
    x_sigma=2.0,
    epsilon_sigma=1.0,
    beta_0_null=0.0,
    beta_1_null=0.0,
    threshold_c=2.0,
    random_seed=42
):
    if num_samples < 3:
        print("Wrong setup: each OLS fit needs at least 3 samples to estimate residual variance.")
        return

    rng = np.random.default_rng(random_seed)

    beta_0_hats = np.empty(num_times)
    beta_1_hats = np.empty(num_times)
    se_beta_0_hats = np.empty(num_times)
    se_beta_1_hats = np.empty(num_times)

    example_X = None
    example_Y = None
    example_Y_estimated = None

    for t in range(num_times):
        X = rng.normal(loc=0.0, scale=x_sigma, size=num_samples)
        epsilon = rng.normal(loc=0.0, scale=epsilon_sigma, size=num_samples)
        Y = beta_0 + beta_1 * X + epsilon

        X_mean = X.mean()
        Y_mean = Y.mean()

        Sxx = np.sum((X - X_mean) ** 2)

        if Sxx == 0:
            print("Wrong setup: OLS cannot estimate a slope because Var(X) is zero.")
            return

        beta_1_hat = np.sum((X - X_mean) * (Y - Y_mean)) / Sxx
        beta_0_hat = Y_mean - beta_1_hat * X_mean

        Y_estimated = beta_0_hat + beta_1_hat * X
        residuals = Y - Y_estimated

        sigma_squared_residual = np.sum(residuals ** 2) / (num_samples - 2)

        se_beta_1_hat = np.sqrt(sigma_squared_residual / Sxx)

        se_beta_0_hat = np.sqrt(
            sigma_squared_residual
            * (
                (1 / num_samples)
                + (X_mean ** 2 / Sxx)
            )
        )

        beta_0_hats[t] = beta_0_hat
        beta_1_hats[t] = beta_1_hat
        se_beta_0_hats[t] = se_beta_0_hat
        se_beta_1_hats[t] = se_beta_1_hat

        if t == 0:
            example_X = X
            example_Y = Y
            example_Y_estimated = Y_estimated

    beta_0_hat_mean = beta_0_hats.mean()
    beta_1_hat_mean = beta_1_hats.mean()

    beta_0_hat_sample_deviation = beta_0_hats.std(ddof=1)
    beta_1_hat_sample_deviation = beta_1_hats.std(ddof=1)

    mean_se_beta_0 = se_beta_0_hats.mean()
    mean_se_beta_1 = se_beta_1_hats.mean()

    beta_1_t_statistics = (beta_1_hats - beta_1_null) / se_beta_1_hats

    beta_1_reject_upper = beta_1_t_statistics > threshold_c

    beta_1_num_reject_upper = np.sum(beta_1_reject_upper)

    beta_1_percent_reject_upper = 100 * beta_1_num_reject_upper / num_times

    beta_0_upper_threshold = beta_0_null + threshold_c * mean_se_beta_0
    beta_0_lower_threshold = beta_0_null - threshold_c * mean_se_beta_0

    beta_1_upper_threshold = beta_1_null + threshold_c * mean_se_beta_1
    beta_1_lower_threshold = beta_1_null - threshold_c * mean_se_beta_1

    beta_0_num_above_threshold = np.sum(beta_0_hats > beta_0_upper_threshold)
    beta_0_num_below_threshold = np.sum(beta_0_hats < beta_0_lower_threshold)

    beta_1_num_above_threshold = np.sum(beta_1_hats > beta_1_upper_threshold)
    beta_1_num_below_threshold = np.sum(beta_1_hats < beta_1_lower_threshold)

    beta_0_num_outside_thresholds = np.sum(
        (beta_0_hats > beta_0_upper_threshold)
        | (beta_0_hats < beta_0_lower_threshold)
    )

    beta_1_num_outside_thresholds = np.sum(
        (beta_1_hats > beta_1_upper_threshold)
        | (beta_1_hats < beta_1_lower_threshold)
    )

    beta_0_percent_above_threshold = 100 * beta_0_num_above_threshold / num_times
    beta_0_percent_below_threshold = 100 * beta_0_num_below_threshold / num_times
    beta_0_percent_outside_thresholds = 100 * beta_0_num_outside_thresholds / num_times

    beta_1_percent_above_threshold = 100 * beta_1_num_above_threshold / num_times
    beta_1_percent_below_threshold = 100 * beta_1_num_below_threshold / num_times
    beta_1_percent_outside_thresholds = 100 * beta_1_num_outside_thresholds / num_times

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    beta_0_plot_sd = max(mean_se_beta_0, beta_0_hat_sample_deviation)

    beta_0_x_min = min(
        beta_0_hats.min(),
        beta_0_null - 4 * beta_0_plot_sd,
        beta_0_lower_threshold
    )

    beta_0_x_max = max(
        beta_0_hats.max(),
        beta_0_null + 4 * beta_0_plot_sd,
        beta_0_upper_threshold
    )

    beta_0_bins = np.linspace(beta_0_x_min, beta_0_x_max, 31)

    beta_0_counts, beta_0_bins, _ = axes[0].hist(
        beta_0_hats,
        bins=beta_0_bins,
        edgecolor="black",
        alpha=0.55,
        label="Simulated beta_0 estimates",
        zorder=2
    )

    beta_0_bin_width = beta_0_bins[1] - beta_0_bins[0]

    beta_0_x_values = np.linspace(
        beta_0_x_min,
        beta_0_x_max,
        500
    )

    beta_0_null_pdf = normal_pdf(
        beta_0_x_values,
        beta_0_null,
        mean_se_beta_0
    )

    beta_0_null_pdf_scaled = beta_0_null_pdf * num_times * beta_0_bin_width

    axes[0].fill_between(
        beta_0_x_values,
        beta_0_null_pdf_scaled,
        color="orange",
        alpha=0.35,
        label="Null beta_0 PDF, scaled to frequency",
        zorder=4
    )

    axes[0].plot(
        beta_0_x_values,
        beta_0_null_pdf_scaled,
        color="darkorange",
        linewidth=3,
        alpha=1.0,
        zorder=5
    )

    axes[0].axvline(
        beta_0,
        color="blue",
        linestyle=":",
        linewidth=2,
        label=f"True beta_0 = {beta_0:.3f}",
        zorder=6
    )
    axes[0].axvline(
        beta_0_hat_mean,
        color="black",
        linewidth=2,
        label=f"Mean estimate = {beta_0_hat_mean:.3f}",
        zorder=6
    )
    axes[0].axvline(
        beta_0_hat_mean - beta_0_hat_sample_deviation,
        color="red",
        linestyle="--",
        linewidth=2,
        label="Mean ± sample deviation",
        zorder=6
    )
    axes[0].axvline(
        beta_0_hat_mean + beta_0_hat_sample_deviation,
        color="red",
        linestyle="--",
        linewidth=2,
        zorder=6
    )
    axes[0].axvline(
        beta_0_null,
        color="orange",
        linestyle=":",
        linewidth=2,
        label=f"Null beta_0 = {beta_0_null:.3f}",
        zorder=6
    )
    axes[0].axvline(
        beta_0_upper_threshold,
        color="green",
        linestyle="--",
        linewidth=2,
        label=f"Null + cSE = {beta_0_upper_threshold:.3f}",
        zorder=6
    )
    axes[0].axvline(
        beta_0_lower_threshold,
        color="green",
        linestyle="--",
        linewidth=2,
        label=f"Null - cSE = {beta_0_lower_threshold:.3f}",
        zorder=6
    )
    axes[0].set_title("Sampling Distribution of beta_0 Hat")
    axes[0].set_xlabel("Estimated beta_0")
    axes[0].set_ylabel("Frequency")
    axes[0].legend()
    axes[0].grid(alpha=0.25)

    beta_1_plot_sd = max(mean_se_beta_1, beta_1_hat_sample_deviation)

    beta_1_x_min = min(
        beta_1_hats.min(),
        beta_1_null - 4 * beta_1_plot_sd,
        beta_1_lower_threshold
    )

    beta_1_x_max = max(
        beta_1_hats.max(),
        beta_1_null + 4 * beta_1_plot_sd,
        beta_1_upper_threshold
    )

    beta_1_bins = np.linspace(beta_1_x_min, beta_1_x_max, 31)

    beta_1_counts, beta_1_bins, _ = axes[1].hist(
        beta_1_hats,
        bins=beta_1_bins,
        edgecolor="black",
        alpha=0.55,
        label="Simulated beta_1 estimates",
        zorder=2
    )

    beta_1_bin_width = beta_1_bins[1] - beta_1_bins[0]

    beta_1_x_values = np.linspace(
        beta_1_x_min,
        beta_1_x_max,
        500
    )

    beta_1_null_pdf = normal_pdf(
        beta_1_x_values,
        beta_1_null,
        mean_se_beta_1
    )

    beta_1_null_pdf_scaled = beta_1_null_pdf * num_times * beta_1_bin_width

    axes[1].fill_between(
        beta_1_x_values,
        beta_1_null_pdf_scaled,
        color="orange",
        alpha=0.35,
        label="Null beta_1 PDF, scaled to frequency",
        zorder=4
    )

    axes[1].plot(
        beta_1_x_values,
        beta_1_null_pdf_scaled,
        color="darkorange",
        linewidth=3,
        alpha=1.0,
        zorder=5
    )

    axes[1].axvline(
        beta_1,
        color="blue",
        linestyle=":",
        linewidth=2,
        label=f"True beta_1 = {beta_1:.3f}",
        zorder=6
    )
    axes[1].axvline(
        beta_1_hat_mean,
        color="black",
        linewidth=2,
        label=f"Mean estimate = {beta_1_hat_mean:.3f}",
        zorder=6
    )
    axes[1].axvline(
        beta_1_hat_mean - beta_1_hat_sample_deviation,
        color="red",
        linestyle="--",
        linewidth=2,
        label="Mean ± sample deviation",
        zorder=6
    )
    axes[1].axvline(
        beta_1_hat_mean + beta_1_hat_sample_deviation,
        color="red",
        linestyle="--",
        linewidth=2,
        zorder=6
    )
    axes[1].axvline(
        beta_1_null,
        color="orange",
        linestyle=":",
        linewidth=2,
        label=f"Null beta_1 = {beta_1_null:.3f}",
        zorder=6
    )
    axes[1].axvline(
        beta_1_upper_threshold,
        color="green",
        linestyle="--",
        linewidth=2,
        label=f"Null + cSE = {beta_1_upper_threshold:.3f}",
        zorder=6
    )
    axes[1].axvline(
        beta_1_lower_threshold,
        color="green",
        linestyle="--",
        linewidth=2,
        label=f"Null - cSE = {beta_1_lower_threshold:.3f}",
        zorder=6
    )
    axes[1].set_title("Sampling Distribution of beta_1 Hat")
    axes[1].set_xlabel("Estimated beta_1")
    axes[1].set_ylabel("Frequency")
    axes[1].legend()
    axes[1].grid(alpha=0.25)

    plt.tight_layout()
    plt.show()

    if num_samples <= 20:
        print("Example first OLS sample:")
        print()
        print(f"{'row':>3} {'X':>12} {'Y':>12} {'Y_estimated':>14}")
        print("-" * 45)

        for i in range(num_samples):
            print(
                f"{i + 1:>3} "
                f"{example_X[i]:>12.6f} "
                f"{example_Y[i]:>12.6f} "
                f"{example_Y_estimated[i]:>14.6f}"
            )

        print()
    else:
        print(
            "Example row display skipped because rows are only printed when "
            "num_samples <= 20."
        )
        print()

    print(f"num_samples per OLS fit = {num_samples}")
    print(f"num_times repeated OLS fits = {num_times}")
    print(f"total simulated observations = {num_samples * num_times}")
    print(f"random seed = {random_seed}")
    print()
    print(f"true beta_0 = {beta_0:.6f}")
    print(f"true beta_1 = {beta_1:.6f}")
    print(f"X is drawn from Normal(0, {x_sigma:.6f})")
    print(f"epsilon is drawn from Normal(0, {epsilon_sigma:.6f})")
    print()
    print("beta_0 estimates:")
    print(f"  mean(beta_0_hat) = {beta_0_hat_mean:.6f}")
    print(f"  sample deviation of beta_0_hat values = {beta_0_hat_sample_deviation:.6f}")
    print(f"  mean estimated standard error of beta_0_hat = {mean_se_beta_0:.6f}")
    print()
    print("beta_1 estimates:")
    print(f"  mean(beta_1_hat) = {beta_1_hat_mean:.6f}")
    print(f"  sample deviation of beta_1_hat values = {beta_1_hat_sample_deviation:.6f}")
    print(f"  mean estimated standard error of beta_1_hat = {mean_se_beta_1:.6f}")
    print()
    print("Frequentist-style threshold settings")
    print(f"  threshold c = {threshold_c:.6f}")
    print(f"  beta_0 null value = {beta_0_null:.6f}")
    print(f"  beta_1 null value = {beta_1_null:.6f}")
    print()
    print("Threshold analysis for beta_0")
    print(f"  beta_0 lower threshold = null - cSE = {beta_0_lower_threshold:.6f}")
    print(f"  beta_0 upper threshold = null + cSE = {beta_0_upper_threshold:.6f}")
    print(f"  number below lower threshold = {beta_0_num_below_threshold}")
    print(f"  number above upper threshold = {beta_0_num_above_threshold}")
    print(f"  number outside two-sided thresholds = {beta_0_num_outside_thresholds}")
    print(f"  percent below lower threshold = {beta_0_percent_below_threshold:.2f}%")
    print(f"  percent above upper threshold = {beta_0_percent_above_threshold:.2f}%")
    print(f"  percent outside two-sided thresholds = {beta_0_percent_outside_thresholds:.2f}%")
    print()
    print("Threshold analysis for beta_1")
    print(f"  beta_1 lower threshold = null - cSE = {beta_1_lower_threshold:.6f}")
    print(f"  beta_1 upper threshold = null + cSE = {beta_1_upper_threshold:.6f}")
    print(f"  number below lower threshold = {beta_1_num_below_threshold}")
    print(f"  number above upper threshold = {beta_1_num_above_threshold}")
    print(f"  number outside two-sided thresholds = {beta_1_num_outside_thresholds}")
    print(f"  percent below lower threshold = {beta_1_percent_below_threshold:.2f}%")
    print(f"  percent above upper threshold = {beta_1_percent_above_threshold:.2f}%")
    print(f"  percent outside two-sided thresholds = {beta_1_percent_outside_thresholds:.2f}%")
    print()
    print(
        "For beta_1, each run estimates the usual OLS standard error as "
        "sqrt(sigma_squared_residual / Sxx), where "
        "Sxx = sum((X_i - mean(X))^2)."
    )
    print()
    print("Actual one-sided upper-tail t-test for beta_1")
    print(f"  H0: beta_1 = {beta_1_null:.6f}")
    print(f"  H1: beta_1 > {beta_1_null:.6f}")
    print(f"  threshold c = {threshold_c:.6f}")
    print(f"  number rejected = {beta_1_num_reject_upper}")
    print(f"  percent rejected = {beta_1_percent_reject_upper:.2f}%")
    print()
    print(
        "Interpretation: threshold_c controls the critical standard-error "
        "multiple. For an approximate one-sided 5% test, use c = 1.645. "
        "For an approximate two-sided 5% test, use c = 1.96. "
        "The percent above the upper threshold is an empirical one-sided "
        "power estimate for a positive effect. The percent outside the lower "
        "and upper thresholds is an empirical two-sided detection rate."
    )


interact(
    repeated_ols_simulator,
    num_samples=IntSlider(
        value=10,
        min=3,
        max=200,
        step=1,
        description="num_samples"
    ),
    num_times=IntSlider(
        value=1000,
        min=2,
        max=10000,
        step=100,
        description="num_times"
    ),
    beta_0=FloatSlider(
        value=1.0,
        min=-10.0,
        max=10.0,
        step=0.1,
        description="beta_0"
    ),
    beta_1=FloatSlider(
        value=2.0,
        min=-10.0,
        max=10.0,
        step=0.1,
        description="beta_1"
    ),
    x_sigma=FloatSlider(
        value=2.0,
        min=0.1,
        max=10.0,
        step=0.1,
        description="X sigma"
    ),
    epsilon_sigma=FloatSlider(
        value=1.0,
        min=0.1,
        max=10.0,
        step=0.1,
        description="eps sigma"
    ),
    beta_0_null=FloatSlider(
        value=0.0,
        min=-10.0,
        max=10.0,
        step=0.1,
        description="b0 null"
    ),
    beta_1_null=FloatSlider(
        value=0.0,
        min=-10.0,
        max=10.0,
        step=0.1,
        description="b1 null"
    ),
    threshold_c=FloatSlider(
        value=2.0,
        min=0.0,
        max=5.0,
        step=0.05,
        description="c"
    ),
    random_seed=IntSlider(
        value=42,
        min=0,
        max=10000,
        step=1,
        description="seed"
    )
);

X = (eps_X: Var(X))
Y = beta_1 * X + beta_0 + (eps_Y: error term -> residual)
# sqrt(residual**2 / (Var(X) * num_samples))

interactive(children=(IntSlider(value=10, description='num_samples', max=200, min=3), IntSlider(value=1000, de…

# Bootstrap simulation

In [17]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider


def ols_fit(X, Y):
    X_mean = X.mean()
    Y_mean = Y.mean()

    Sxx = np.sum((X - X_mean) ** 2)

    if Sxx == 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    beta_1_hat = np.sum((X - X_mean) * (Y - Y_mean)) / Sxx
    beta_0_hat = Y_mean - beta_1_hat * X_mean

    Y_hat = beta_0_hat + beta_1_hat * X
    residuals = Y - Y_hat

    n = len(X)

    if n <= 2:
        return beta_0_hat, beta_1_hat, np.nan, np.nan, np.nan

    sigma_squared_residual = np.sum(residuals ** 2) / (n - 2)

    se_beta_1_hat = np.sqrt(sigma_squared_residual / Sxx)

    se_beta_0_hat = np.sqrt(
        sigma_squared_residual
        * (
            (1 / n)
            + (X_mean ** 2 / Sxx)
        )
    )

    return beta_0_hat, beta_1_hat, se_beta_0_hat, se_beta_1_hat, sigma_squared_residual


def bootstrap_ols_simulator(
    num_samples=30,
    beta_0=1.0,
    beta_1=0.5,
    x_sigma=2.0,
    epsilon_sigma=1.0,
    num_bootstrap=1000,
    threshold_c=2.0,
    random_seed=42
):
    if num_samples < 3:
        print("Wrong setup: OLS needs at least 3 samples.")
        return

    if num_bootstrap < 2:
        print("Wrong setup: need at least 2 bootstrap samples.")
        return

    rng = np.random.default_rng(random_seed)

    X = rng.normal(loc=0.0, scale=x_sigma, size=num_samples)
    epsilon = rng.normal(loc=0.0, scale=epsilon_sigma, size=num_samples)
    Y = beta_0 + beta_1 * X + epsilon

    (
        beta_0_hat,
        beta_1_hat,
        se_beta_0_hat,
        se_beta_1_hat,
        sigma_squared_residual
    ) = ols_fit(X, Y)

    if np.isnan(beta_1_hat):
        print("Wrong setup: OLS cannot estimate a slope because Var(X) is zero.")
        return

    Y_hat = beta_0_hat + beta_1_hat * X
    residuals_full = Y - Y_hat

    beta_0_boot_row = np.empty(num_bootstrap)
    beta_1_boot_row = np.empty(num_bootstrap)

    beta_0_boot_resid_full = np.empty(num_bootstrap)
    beta_1_boot_resid_full = np.empty(num_bootstrap)

    beta_0_boot_null = np.empty(num_bootstrap)
    beta_1_boot_null = np.empty(num_bootstrap)

    for b in range(num_bootstrap):
        row_indices = rng.integers(
            low=0,
            high=num_samples,
            size=num_samples
        )

        X_star_row = X[row_indices]
        Y_star_row = Y[row_indices]

        (
            beta_0_star_row,
            beta_1_star_row,
            _,
            _,
            _
        ) = ols_fit(X_star_row, Y_star_row)

        beta_0_boot_row[b] = beta_0_star_row
        beta_1_boot_row[b] = beta_1_star_row

        residual_indices_full = rng.integers(
            low=0,
            high=num_samples,
            size=num_samples
        )

        residuals_star_full = residuals_full[residual_indices_full]

        Y_star_resid_full = Y_hat + residuals_star_full

        (
            beta_0_star_resid_full,
            beta_1_star_resid_full,
            _,
            _,
            _
        ) = ols_fit(X, Y_star_resid_full)

        beta_0_boot_resid_full[b] = beta_0_star_resid_full
        beta_1_boot_resid_full[b] = beta_1_star_resid_full

        null_fitted_value = Y.mean()
        null_residuals = Y - null_fitted_value

        residual_indices_null = rng.integers(
            low=0,
            high=num_samples,
            size=num_samples
        )

        residuals_star_null = null_residuals[residual_indices_null]

        Y_star_null = null_fitted_value + residuals_star_null

        (
            beta_0_star_null,
            beta_1_star_null,
            _,
            _,
            _
        ) = ols_fit(X, Y_star_null)

        beta_0_boot_null[b] = beta_0_star_null
        beta_1_boot_null[b] = beta_1_star_null

    beta_0_boot_row = beta_0_boot_row[~np.isnan(beta_0_boot_row)]
    beta_1_boot_row = beta_1_boot_row[~np.isnan(beta_1_boot_row)]

    beta_0_boot_resid_full = beta_0_boot_resid_full[~np.isnan(beta_0_boot_resid_full)]
    beta_1_boot_resid_full = beta_1_boot_resid_full[~np.isnan(beta_1_boot_resid_full)]

    beta_0_boot_null = beta_0_boot_null[~np.isnan(beta_0_boot_null)]
    beta_1_boot_null = beta_1_boot_null[~np.isnan(beta_1_boot_null)]

    bootstrap_se_beta_0_row = beta_0_boot_row.std(ddof=1)
    bootstrap_se_beta_1_row = beta_1_boot_row.std(ddof=1)

    bootstrap_se_beta_0_resid_full = beta_0_boot_resid_full.std(ddof=1)
    bootstrap_se_beta_1_resid_full = beta_1_boot_resid_full.std(ddof=1)

    bootstrap_se_beta_0_null = beta_0_boot_null.std(ddof=1)
    bootstrap_se_beta_1_null = beta_1_boot_null.std(ddof=1)

    beta_1_null = 0.0

    beta_1_upper_threshold_parametric = beta_1_null + threshold_c * se_beta_1_hat
    beta_1_lower_threshold_parametric = beta_1_null - threshold_c * se_beta_1_hat

    beta_1_upper_threshold_boot_null = beta_1_null + threshold_c * bootstrap_se_beta_1_null
    beta_1_lower_threshold_boot_null = beta_1_null - threshold_c * bootstrap_se_beta_1_null

    beta_1_above_parametric = beta_1_hat > beta_1_upper_threshold_parametric
    beta_1_below_parametric = beta_1_hat < beta_1_lower_threshold_parametric
    beta_1_outside_parametric = beta_1_above_parametric or beta_1_below_parametric

    beta_1_above_boot_null = beta_1_hat > beta_1_upper_threshold_boot_null
    beta_1_below_boot_null = beta_1_hat < beta_1_lower_threshold_boot_null
    beta_1_outside_boot_null = beta_1_above_boot_null or beta_1_below_boot_null

    bootstrap_null_percent_above_observed = (
        100 * np.mean(beta_1_boot_null >= beta_1_hat)
    )

    bootstrap_null_percent_as_extreme_two_sided = (
        100 * np.mean(np.abs(beta_1_boot_null) >= abs(beta_1_hat))
    )

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].scatter(X, Y, alpha=0.8, edgecolor="black")
    x_grid = np.linspace(X.min(), X.max(), 200)
    y_grid_true = beta_0 + beta_1 * x_grid
    y_grid_hat = beta_0_hat + beta_1_hat * x_grid
    y_grid_null = Y.mean() + 0 * x_grid

    axes[0, 0].plot(
        x_grid,
        y_grid_true,
        color="blue",
        linestyle=":",
        linewidth=2,
        label=f"True line: beta_1 = {beta_1:.3f}"
    )
    axes[0, 0].plot(
        x_grid,
        y_grid_hat,
        color="black",
        linewidth=2,
        label=f"OLS line: beta_1_hat = {beta_1_hat:.3f}"
    )
    axes[0, 0].plot(
        x_grid,
        y_grid_null,
        color="orange",
        linestyle="--",
        linewidth=2,
        label="Null line: slope = 0"
    )
    axes[0, 0].set_title("Observed Sample and Fitted Lines")
    axes[0, 0].set_xlabel("X")
    axes[0, 0].set_ylabel("Y")
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.25)

    axes[0, 1].hist(
        beta_1_boot_row,
        bins=30,
        edgecolor="black",
        alpha=0.75
    )
    axes[0, 1].axvline(
        beta_1_hat,
        color="black",
        linewidth=2,
        label=f"Observed beta_1_hat = {beta_1_hat:.3f}"
    )
    axes[0, 1].axvline(
        beta_1_hat - threshold_c * bootstrap_se_beta_1_row,
        color="red",
        linestyle="--",
        linewidth=2,
        label=f"Observed ± c bootstrap SE"
    )
    axes[0, 1].axvline(
        beta_1_hat + threshold_c * bootstrap_se_beta_1_row,
        color="red",
        linestyle="--",
        linewidth=2
    )
    axes[0, 1].set_title("Row Bootstrap Distribution of beta_1 Hat")
    axes[0, 1].set_xlabel("Bootstrap beta_1_hat")
    axes[0, 1].set_ylabel("Frequency")
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.25)

    axes[1, 0].hist(
        beta_1_boot_resid_full,
        bins=30,
        edgecolor="black",
        alpha=0.75
    )
    axes[1, 0].axvline(
        beta_1_hat,
        color="black",
        linewidth=2,
        label=f"Observed beta_1_hat = {beta_1_hat:.3f}"
    )
    axes[1, 0].axvline(
        beta_1_hat - threshold_c * bootstrap_se_beta_1_resid_full,
        color="red",
        linestyle="--",
        linewidth=2,
        label=f"Observed ± c bootstrap SE"
    )
    axes[1, 0].axvline(
        beta_1_hat + threshold_c * bootstrap_se_beta_1_resid_full,
        color="red",
        linestyle="--",
        linewidth=2
    )
    axes[1, 0].set_title("Residual Bootstrap Around Full Model")
    axes[1, 0].set_xlabel("Bootstrap beta_1_hat")
    axes[1, 0].set_ylabel("Frequency")
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.25)

    axes[1, 1].hist(
        beta_1_boot_null,
        bins=30,
        edgecolor="black",
        alpha=0.75
    )
    axes[1, 1].axvline(
        beta_1_null,
        color="orange",
        linestyle=":",
        linewidth=2,
        label="Null beta_1 = 0"
    )
    axes[1, 1].axvline(
        beta_1_hat,
        color="black",
        linewidth=2,
        label=f"Observed beta_1_hat = {beta_1_hat:.3f}"
    )
    axes[1, 1].axvline(
        beta_1_upper_threshold_boot_null,
        color="green",
        linestyle="--",
        linewidth=2,
        label=f"Null + c bootstrap SE = {beta_1_upper_threshold_boot_null:.3f}"
    )
    axes[1, 1].axvline(
        beta_1_lower_threshold_boot_null,
        color="green",
        linestyle="--",
        linewidth=2,
        label=f"Null - c bootstrap SE = {beta_1_lower_threshold_boot_null:.3f}"
    )
    axes[1, 1].set_title("Residual Bootstrap Under Null Model beta_1 = 0")
    axes[1, 1].set_xlabel("Null-bootstrap beta_1_hat")
    axes[1, 1].set_ylabel("Frequency")
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.25)

    plt.tight_layout()
    plt.show()

    if num_samples <= 30:
        print("Observed sample:")
        print()
        print(f"{'row':>3} {'X':>12} {'Y':>12} {'Y_hat':>12} {'residual':>12}")
        print("-" * 58)

        for i in range(num_samples):
            print(
                f"{i + 1:>3} "
                f"{X[i]:>12.6f} "
                f"{Y[i]:>12.6f} "
                f"{Y_hat[i]:>12.6f} "
                f"{residuals_full[i]:>12.6f}"
            )

        print()
    else:
        print(
            "Observed row display skipped because rows are only printed when "
            "num_samples <= 30."
        )
        print()

    print(f"num_samples = {num_samples}")
    print(f"num_bootstrap = {num_bootstrap}")
    print(f"random_seed = {random_seed}")
    print()
    print("Data-generating process")
    print(f"  Y = beta_0 + beta_1 * X + epsilon")
    print(f"  true beta_0 = {beta_0:.6f}")
    print(f"  true beta_1 = {beta_1:.6f}")
    print(f"  X is drawn from Normal(0, {x_sigma:.6f})")
    print(f"  epsilon is drawn from Normal(0, {epsilon_sigma:.6f})")
    print()
    print("Observed OLS fit")
    print(f"  beta_0_hat = {beta_0_hat:.6f}")
    print(f"  beta_1_hat = {beta_1_hat:.6f}")
    print(f"  parametric SE(beta_0_hat) = {se_beta_0_hat:.6f}")
    print(f"  parametric SE(beta_1_hat) = {se_beta_1_hat:.6f}")
    print(f"  residual variance estimate = {sigma_squared_residual:.6f}")
    print()
    print("Bootstrap standard errors")
    print(f"  row bootstrap SE(beta_0_hat) = {bootstrap_se_beta_0_row:.6f}")
    print(f"  row bootstrap SE(beta_1_hat) = {bootstrap_se_beta_1_row:.6f}")
    print(f"  residual bootstrap around full model SE(beta_0_hat) = {bootstrap_se_beta_0_resid_full:.6f}")
    print(f"  residual bootstrap around full model SE(beta_1_hat) = {bootstrap_se_beta_1_resid_full:.6f}")
    print(f"  residual bootstrap under null SE(beta_0_hat) = {bootstrap_se_beta_0_null:.6f}")
    print(f"  residual bootstrap under null SE(beta_1_hat) = {bootstrap_se_beta_1_null:.6f}")
    print()
    print("Null hypothesis")
    print("  H0: beta_1 = 0")
    print("  This does not mean beta_0 = 0.")
    print("  Under the null model, Y = beta_0 + epsilon.")
    print("  The null fitted line is horizontal at mean(Y).")
    print()
    print("Frequentist-style threshold using c")
    print(f"  threshold_c = {threshold_c:.6f}")
    print()
    print("Using parametric OLS SE")
    print(f"  lower threshold = 0 - c * SE(beta_1_hat) = {beta_1_lower_threshold_parametric:.6f}")
    print(f"  upper threshold = 0 + c * SE(beta_1_hat) = {beta_1_upper_threshold_parametric:.6f}")
    print(f"  beta_1_hat below lower threshold? {beta_1_below_parametric}")
    print(f"  beta_1_hat above upper threshold? {beta_1_above_parametric}")
    print(f"  beta_1_hat outside two-sided thresholds? {beta_1_outside_parametric}")
    print()
    print("Using null residual-bootstrap SE")
    print(f"  lower threshold = 0 - c * bootstrap SE = {beta_1_lower_threshold_boot_null:.6f}")
    print(f"  upper threshold = 0 + c * bootstrap SE = {beta_1_upper_threshold_boot_null:.6f}")
    print(f"  beta_1_hat below lower threshold? {beta_1_below_boot_null}")
    print(f"  beta_1_hat above upper threshold? {beta_1_above_boot_null}")
    print(f"  beta_1_hat outside two-sided thresholds? {beta_1_outside_boot_null}")
    print()
    print("Approximate bootstrap p-value style summaries")
    print(
        "  percent of null-bootstrap beta_1_hat values >= observed beta_1_hat "
        f"= {bootstrap_null_percent_above_observed:.2f}%"
    )
    print(
        "  percent of null-bootstrap beta_1_hat values at least as extreme as observed "
        f"= {bootstrap_null_percent_as_extreme_two_sided:.2f}%"
    )
    print()
    print("Interpretation")
    print(
        "  Row bootstrap and full-model residual bootstrap estimate the standard "
        "error of beta_1_hat around the observed fitted relationship."
    )
    print(
        "  Null residual bootstrap estimates what beta_1_hat would look like if "
        "beta_1 were truly zero but Y still had noise around a horizontal line."
    )
    print(
        "  If beta_1 is nonzero, the observed beta_1_hat may fall beyond the "
        "null + cSE threshold. That is the basic detection idea."
    )


interact(
    bootstrap_ols_simulator,
    num_samples=IntSlider(
        value=30,
        min=3,
        max=200,
        step=1,
        description="n"
    ),
    beta_0=FloatSlider(
        value=1.0,
        min=-10.0,
        max=10.0,
        step=0.1,
        description="beta_0"
    ),
    beta_1=FloatSlider(
        value=0.5,
        min=-5.0,
        max=5.0,
        step=0.1,
        description="beta_1"
    ),
    x_sigma=FloatSlider(
        value=2.0,
        min=0.1,
        max=10.0,
        step=0.1,
        description="X sigma"
    ),
    epsilon_sigma=FloatSlider(
        value=1.0,
        min=0.1,
        max=10.0,
        step=0.1,
        description="eps sigma"
    ),
    num_bootstrap=IntSlider(
        value=1000,
        min=100,
        max=5000,
        step=100,
        description="bootstrap"
    ),
    threshold_c=FloatSlider(
        value=2.0,
        min=0.0,
        max=5.0,
        step=0.05,
        description="c"
    ),
    random_seed=IntSlider(
        value=42,
        min=0,
        max=10000,
        step=1,
        description="seed"
    )
);

# UPPER RIGHT:
# If the observed dataset is a stand-in for the population, how much would the estimated slope 
# vary from sample to sample?

# LOWER LEFT:
# If the fitted regression line were the true relationship, how much would the estimated slope vary 
# because of random noise?

# LOWER RIGHT:
# If there were truly no slope, how large could the estimated slope look just because of random noise?
# Q: Would a slope this large be common if the true slope were zero?

interactive(children=(IntSlider(value=30, description='n', max=200, min=3), FloatSlider(value=1.0, description…